Create the Gold Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS openskyreal.gold;

Read Silver Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
checkpoint_path3 = "abfss://real@uralibootcamp.dfs.core.windows.net/checkpoint/gold_Flights"
checkpoint_path_gold_delta = (
"abfss://real@uralibootcamp.dfs.core.windows.net/checkpoints/gold_delta"
)
checkpoint_path_gold_view = (
"abfss://real@uralibootcamp.dfs.core.windows.net/checkpoints1/gold_view"
)

silver_current = (

spark.readStream
.table("openskyreal.silver.flights")

.filter(
    col("is_current") == True
)

)

Country Traffic 

Use for:

Active flights per country
Top 10 countries

In [0]:
from pyspark.sql.functions import *

country_traffic_df1 = (
    silver_current 
    .groupBy("origin_country")
    .agg(
        approx_count_distinct("icao24").alias("active_flights")
    )
    .orderBy(desc("active_flights"))
)

display(country_traffic_df1, checkpointLocation = checkpoint_path_gold_view)

In [0]:
(
country_traffic_df1.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_path_gold_delta)
    .toTable("openskyreal.gold.country_traffic")
)

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


gold_query = (
    country_traffic_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path3
    )
    .outputMode("complete")
    .trigger(
        processingTime="2 minutes"
    )
    .option(
        "mergeSchema",
        "true"
    )
    .option(
        "path",
        checkpoint_path3
    )
    .start()
)

Speed Analytics 

Use for:

Average speed
Fastest aircraft
Speed distribution

In [0]:
speed_analysis_df1 = (
    silver_current 
    .groupBy("origin_country")
    .agg(
        avg("speed_kmh").alias("average_speed"),
        max("speed_kmh").alias("fastest_speed")
    )
)

display(speed_analysis_df1,checkpointLocation = checkpoint_path_gold_view)

In [0]:
(speed_analysis_df1.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_path_gold_delta)
    .toTable("openskyreal.gold.speed_analysis")
)


In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)


gold_query = (
    speed_analysis_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path3
    )
    .outputMode("complete")
    .trigger(
        processingTime="2 minutes"
    )
    .option(
        "mergeSchema",
        "true"
    )
    .option(
        "path",
        checkpoint_path3
    )
    .start()
)

Flight Direction Analysis 

Use for:

Direction heatmap
Air corridor detection

In [0]:
direction_analysis_df1 = (
    silver_current 
    .groupBy("direction")
    .agg(
        count("*").alias("flight_count")
    )
)

display(direction_analysis_df1,checkpointLocation=checkpoint_path_gold_view)

In [0]:
(direction_analysis_df1.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_path_gold_delta)
    .toTable("openskyreal.gold.direction_analysis")
)

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)

gold_query = (
    direction_analysis_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path3
    )
    .outputMode("complete")
    .trigger(
        processingTime="2 minutes"
    )
    .option(
        "mergeSchema",
        "true"
    )
    .option(
        "path",
        checkpoint_path3
    )
    .start()
)

Aircraft Anomaly 

Use:

Sudden speed changes
Ground aircraft movement

In [0]:
anomaly_summary_df1 = (
    silver_current 
    .groupBy("origin_country")
    .agg(
        sum("speed_anomaly_flag").alias("speed_anomalies"),
        sum("ground_movement_flag").alias("ground_movements")
    )
)

display(anomaly_summary_df1,checkpointLocation=checkpoint_path_gold_view)

In [0]:
(anomaly_summary_df1.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation",checkpoint_path_gold_delta)
    .toTable("openskyreal.gold.anomaly_summary")
)

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)

gold_query = (
    anomaly_summary_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path3
    )
    .outputMode("complete")
    .trigger(
        processingTime="2 minutes"
    )
    .option(
        "mergeSchema",
        "true"
    )
    .option(
        "path",
        checkpoint_path3
    )
    .start()
)

Airspace Congestion 

Use:

Aircraft density
Busy airspace detection

In [0]:
airspace_congestion_df1 = (
    silver_current 
    .groupBy("region")
    .agg(
        approx_count_distinct("icao24")
        .alias("aircraft_density")
    )
)

display(airspace_congestion_df1,checkpointLocation=checkpoint_path_gold_view)

In [0]:
(airspace_congestion_df1.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_path_gold_delta)
    .toTable("openskyreal.gold.airspace_congestion")
)

In [0]:
gold_path = (
"abfss://real@uralibootcamp.dfs.core.windows.net/gold/flights"
)

gold_query = (
    airspace_congestion_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path3
    )
    .outputMode("complete")
    .trigger(
        processingTime="2 minutes"
    )
    .option(
        "mergeSchema",
        "true"
    )
    .option(
        "path",
        checkpoint_path3
    )
    .start()
)

In [0]:
%sql
SHOW TABLES IN openskyreal.gold;

In [0]:
%sql
SELECT *
FROM openskyreal.gold.country_traffic;

In [0]:


)